<a href="https://colab.research.google.com/github/Lokesh66666/CV_workshop/blob/main/Yolo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

YOLO


In [13]:
# Core libraries
import tensorflow as tf            # Main deep learning library
import numpy as np                # For numerical operations (arrays, math)
import matplotlib.pyplot as plt   # For plotting graphs and images

# Dataset
from tensorflow.keras.datasets import cifar10   # CIFAR-10 image dataset

# Model and layers
from tensorflow.keras import layers, models     # To build neural network models

# Pretrained model
from tensorflow.keras.applications import MobileNetV2   # Pretrained CNN model

# Utilities
from sklearn.metrics import classification_report, confusion_matrix  # Evaluation metrics
import seaborn as sns   # For better visualization (like heatmaps)

Load dataset

In [14]:
# Load CIFAR-10 dataset (automatically split into train and test)
(x_train, y_train), (x_test, y_test) = cifar10.load_data()

# x_train → training images
# y_train → labels for training images
# x_test → testing images
# y_test → labels for testing images

# Print shapes to understand the data structure
print("Train shape:", x_train.shape)
# Example: (50000, 32, 32, 3) -> 50k images, each 32x32 with 3 color channels

print("Test shape:", x_test.shape)
# Example: (10000, 32, 32, 3)

Train shape: (50000, 32, 32, 3)
Test shape: (10000, 32, 32, 3)


normalized data

In [15]:
# Normalize pixel values (0–255 -> 0–1)
x_train = x_train / 255.0
x_test = x_test / 255.0


Data augmentation

In [16]:
# Data augmentation helps the model learn better by showing slightly modified images
# This reduces overfitting and improves performance on new data

data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),     # Randomly flips images left ↔ right
    layers.RandomRotation(0.1),          # Slightly rotates images (10% range)
    layers.RandomZoom(0.1)               # Slightly zooms in/out
])

load pritrained model


In [17]:
# Load MobileNetV2 model without the final classification layer (top layers)
base_model = MobileNetV2(input_shape=(32,32,3),
                         include_top=False,       # Remove default output layer
                         weights='imagenet')      # Use pretrained weights from ImageNet

# Freeze the base model so its learned features are not changed during training
base_model.trainable = False


/tmp/ipykernel_12482/3397552793.py:2: UserWarning: `input_shape` is undefined or non-square, or `rows` is not in [96, 128, 160, 192, 224]. Weights for input shape (224, 224) will be loaded as the default.
  base_model = MobileNetV2(input_shape=(32,32,3),


Build Advanced model

In [18]:
# Freeze the base model so its learned features are not changed during training
base_model.trainable = False

# Build Advanced Model
# Build custom model on top of the pretrained base model
model = models.Sequential([
    data_augmentation,              # Apply random transformations to input images
    base_model,                     # Use pretrained MobileNetV2 as feature extractor
    layers.GlobalAveragePooling2D(),# Convert feature maps into a single vector
    layers.BatchNormalization(),    # Normalize features to make training stable
    layers.Dense(128, activation="relu"),  # Fully connected layer for learning patterns
    layers.Dropout(0.5),            # Randomly drop neurons to reduce overfitting
    layers.Dense(10, activation="softmax") # Output layer for 10 classes
])

# Display model architecture
model.summary()

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ sequential_2 (Sequential)       │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_224            │ (None, 1, 1, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ ?                      │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ ?                      │   0 (unbuilt) │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,257,984 (8.61 MB)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 2,257,984 (8.61 MB)